[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/myway0101/see-read-explain-dl/blob/main/notebooks/ch01_dnn_basics.ipynb)

# 1장 심층 신경망의 원리 — 실습 노트북
『보고, 읽고, 설명하는 딥러닝 — Colab에서 직접 만들며 배우는 딥러닝 응용』

이 노트북은 1장 본문에 나오는 코드를 순서대로 실행해 봅니다. 위에서부터 차례로 실행하세요.
- 1.3 활성화 함수 그려 보기
- 1.4 원-핫 인코딩과 소프트맥스
- 1.5 손실 함수 계산하기
- 1.7 드롭아웃
- 1.8 옵티마이저 지정하기

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
print("TensorFlow", tf.__version__)

## 1.3 활성화 함수
시그모이드, 하이퍼볼릭 탄젠트, 렐루, 리키 렐루를 직접 그리고, 각 함수의 기울기(미분값)도 함께 그려 봅니다.

In [ ]:
x = np.linspace(-6, 6, 400)
sigmoid = 1 / (1 + np.exp(-x))
tanh = np.tanh(x)
relu = np.maximum(0, x)
leaky = np.where(x > 0, x, 0.01 * x)

fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
for ax, (name, y) in zip(axes, [("sigmoid", sigmoid), ("tanh", tanh), ("ReLU", relu), ("Leaky ReLU", leaky)]):
    dy = np.gradient(y, x)                     # 수치 미분으로 기울기 계산
    ax.plot(x, y, label=name)
    ax.plot(x, dy, "--", label="derivative")
    ax.set_title(name); ax.grid(alpha=0.3); ax.legend()
plt.tight_layout(); plt.show()

print("시그모이드 기울기의 최댓값:", round(np.gradient(sigmoid, x).max(), 3))
print("tanh 기울기의 최댓값    :", round(np.gradient(tanh, x).max(), 3))

텐서플로가 제공하는 활성화 함수로도 같은 값을 얻을 수 있습니다.

In [ ]:
t = tf.constant([-2.0, -0.5, 0.0, 0.5, 2.0])
print("sigmoid   ", tf.nn.sigmoid(t).numpy().round(3))
print("tanh      ", tf.nn.tanh(t).numpy().round(3))
print("relu      ", tf.nn.relu(t).numpy())
print("leaky_relu", tf.nn.leaky_relu(t, alpha=0.01).numpy())

## 1.4 원-핫 인코딩
붓꽃 품종처럼 이름으로 된 정답을 0과 1로 이루어진 벡터로 바꿉니다. (setosa=0, versicolor=1, virginica=2)

In [ ]:
labels = np.array([0, 1, 2, 0, 2])          # setosa, versicolor, virginica, setosa, virginica
one_hot = tf.keras.utils.to_categorical(labels)
print(one_hot)

## 1.4 소프트맥스
출력층의 값 (1, 1, 3)을 확률로 바꿔 봅니다. 그냥 비율로 나눈 확률과 비교해 보세요.

In [ ]:
a = np.array([1.0, 1.0, 3.0])
print("단순 비율  :", (a / a.sum()).round(3))
print("소프트맥스 :", tf.nn.softmax(a).numpy().round(3))

렐루와 소프트맥스를 사용하는 모델은 다음과 같이 만듭니다.

In [ ]:
from tensorflow.keras import layers
model = tf.keras.Sequential()
model.add(layers.Dense(64, activation='relu'))      # 유닛 64개를 가진 완전연결층
model.add(layers.Dense(64, activation='relu'))
model.add(layers.Dense(10, activation='softmax'))   # 출력 유닛 10개, 소프트맥스
model.build(input_shape=(None, 20))                 # 입력 크기를 정해 주면 구조를 볼 수 있습니다
model.summary()

## 1.5 손실 함수
### 평균 제곱 오차(MSE)

In [ ]:
y_true = np.array([3.0, -0.5, 2.0, 7.0])
y_pred = np.array([2.5,  0.0, 2.0, 8.0])
print("직접 계산 :", np.mean((y_true - y_pred) ** 2))
print("케라스    :", tf.keras.losses.MeanSquaredError()(y_true, y_pred).numpy())

모델을 컴파일할 때는 다음과 같이 지정합니다.

> 학습률은 `learning_rate=` 인자로 지정합니다. 예전 예제의 `SGD(lr=0.1)`처럼 `lr=`로 쓰면 오류가 납니다.

In [ ]:
model.compile(optimizer=tf.keras.optimizers.SGD(learning_rate=0.1), loss='mse')

### 크로스 엔트로피 오차(CE)
정답이 원-핫 벡터일 때, 정답 칸의 예측 확률이 높을수록 손실이 작아집니다.

In [ ]:
t = np.array([[0, 0, 1]], dtype="float32")          # 정답: 세 번째 클래스
good = np.array([[0.1, 0.1, 0.8]], dtype="float32")   # 정답을 높게 예측
bad  = np.array([[0.6, 0.3, 0.1]], dtype="float32")   # 정답을 낮게 예측
cce = tf.keras.losses.CategoricalCrossentropy()
print("잘 맞힌 예측   :", round(float(cce(t, good)), 3), "  (= -log 0.8 =", round(-np.log(0.8), 3), ")")
print("잘못 맞힌 예측 :", round(float(cce(t, bad)), 3),  "  (= -log 0.1 =", round(-np.log(0.1), 3), ")")

bce = tf.keras.losses.BinaryCrossentropy()
print("이진 CE (정답 1, 예측 0.9):", round(float(bce([[1.0]], [[0.9]])), 3))

In [ ]:
model.compile(optimizer=tf.keras.optimizers.Adam(0.001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

## 1.7 드롭아웃
학습할 때마다 노드의 20%를 무작위로 끕니다.

In [ ]:
model = tf.keras.models.Sequential([
    tf.keras.layers.Flatten(input_shape=(28, 28)),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.2),           # 드롭아웃 적용
    tf.keras.layers.Dense(10, activation='softmax')
])
model.summary()

드롭아웃은 **학습할 때만** 작동합니다. 같은 입력을 넣고 `training` 값에 따라 출력이 어떻게 달라지는지 확인해 보세요.

In [ ]:
drop = tf.keras.layers.Dropout(0.2)
ones = tf.ones((1, 10))
print("학습할 때  :", drop(ones, training=True).numpy())
print("예측할 때  :", drop(ones, training=False).numpy())

## 1.8 옵티마이저 지정하기
방법 1: 옵티마이저 객체를 만들어 학습률 등을 직접 정한 뒤 컴파일합니다.

In [ ]:
from tensorflow import keras
model = keras.Sequential()
model.add(layers.Dense(64, kernel_initializer='uniform', input_shape=(10,)))
model.add(layers.Activation('softmax'))

opt = keras.optimizers.Adam(learning_rate=0.01)
model.compile(loss='categorical_crossentropy', optimizer=opt)

방법 2: 이름(문자열)으로 지정하면 기본 설정값이 사용됩니다.

In [ ]:
# 이름(문자열)으로 지정하면 기본 설정값이 사용됩니다
model.compile(loss='categorical_crossentropy', optimizer='adam')
print(model.optimizer.__class__.__name__, "learning_rate =", float(model.optimizer.learning_rate))

### 더 해 보기: 경사 하강법이 최솟값을 찾아가는 모습
손실 함수 $L(w) = (w-3)^2$의 최솟값을 경사 하강법으로 찾아봅니다. 학습률을 바꿔 가며 실행해 보세요.

In [ ]:
def gradient_descent(lr, w=0.0, steps=15):
    path = [w]
    for _ in range(steps):
        grad = 2 * (w - 3)          # dL/dw
        w = w - lr * grad           # 가중치 업데이트
        path.append(w)
    return np.array(path)

ws = np.linspace(-1, 7, 200)
plt.figure(figsize=(12, 3.5))
for i, lr in enumerate([0.05, 0.3, 0.95]):
    p = gradient_descent(lr)
    plt.subplot(1, 3, i + 1)
    plt.plot(ws, (ws - 3) ** 2, color="gray")
    plt.plot(p, (p - 3) ** 2, "o-", markersize=4)
    plt.title(f"learning rate = {lr}  (final w = {p[-1]:.2f})")
plt.tight_layout(); plt.show()